<a href="https://colab.research.google.com/github/pr3tzell/CSN/blob/main/Pandas_para_Ciencia_de_Datos_Proyecto_I_Parte_2_(Core).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import pandas as pd
from google.colab import files
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR LOS DATOS
# --------------------------------------------------

print("Selecciona el archivo retail_sales_dataset.csv o retail_sales.csv.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))

# Eliminar posibles espacios al inicio o al final de los encabezados.
datos.columns = datos.columns.str.strip()

# Adaptación automática si se sube el dataset en inglés
if "Product Category" in datos.columns:
    print("\nSe detectó el dataset en inglés. Adaptando columnas...")
    # Mapeamos las columnas del dataset en inglés a las esperadas por el script
    datos = datos.rename(columns={
        "Product Category": "Producto",
        "Price per Unit": "Precio",
        "Quantity": "Ventas"  # O 'Total Amount' según lo que desees analizar como ventas
    })
    # Como este dataset no tiene tienda, creamos una columna simulada para que no falle el script
    datos["Tienda"] = "Tienda Online"

columnas_necesarias = ["Producto", "Tienda", "Ventas", "Precio"]
columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(columnas_faltantes)}. "
        f"El archivo contiene: {', '.join(datos.columns)}. "
        "Comprueba que sea el dataset de esta actividad o uno compatible."
    )

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))


# --------------------------------------------------
# 2. EXPLORACIÓN INICIAL
# --------------------------------------------------

print("\nÚLTIMAS 5 FILAS")
display(datos.tail(5))

# info() muestra la cantidad de registros, las columnas,
# sus tipos de datos y la memoria utilizada.
print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nTIPOS DE DATOS ORIGINALES")
print(datos.dtypes)

# Revisar valores faltantes antes de realizar cambios.
print("\nVALORES FALTANTES POR COLUMNA")
print(datos.isna().sum())


# --------------------------------------------------
# 3. PREPARAR LAS COLUMNAS PARA EL ANÁLISIS
# --------------------------------------------------

# Quitar espacios de los textos y conservar los valores faltantes.
for columna in ["Producto", "Tienda"]:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

# Convertir ventas y precios a números.
# Si algún valor no se puede convertir, quedará como NaN.
# No se reemplaza por cero porque eso cambiaría su significado.
for columna in ["Ventas", "Precio"]:
    valores_originales = datos[columna].copy()

    datos[columna] = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    conversiones_invalidas = (
        valores_originales.notna() & datos[columna].isna()
    ).sum()

    print(
        f"\nValores no numéricos convertidos a NaN en {columna}: "
        f"{conversiones_invalidas}"
    )

print("\nTIPOS DE DATOS DESPUÉS DE LA PREPARACIÓN")
print(datos.dtypes)

# describe() resume las columnas numéricas.
# Los valores faltantes no participan en estos cálculos.
print("\nESTADÍSTICAS DESCRIPTIVAS")
display(datos.describe())

print("\nVALORES FALTANTES DESPUÉS DE LA PREPARACIÓN")
print(datos.isna().sum())


# --------------------------------------------------
# 4. INSPECCIÓN DE PRODUCTOS Y TIENDAS
# --------------------------------------------------

# Contar cuántas filas corresponden a cada producto.
# Este conteo no representa la suma de las ventas.
print("\nCANTIDAD DE REGISTROS POR PRODUCTO")
print(datos["Producto"].value_counts(dropna=False))

# Mostrar las tiendas sin repetirlas.
print("\nTIENDAS DISPONIBLES")
print(datos["Tienda"].unique())


# --------------------------------------------------
# 5. FILTRADO DE DATOS
# --------------------------------------------------

# Seleccionar filas donde Ventas sea mayor a 50.
ventas_mayores_50 = datos[datos["Ventas"] > 50]

print("\nVENTAS MAYORES A 50")
print("Cantidad de registros:", len(ventas_mayores_50))
display(ventas_mayores_50)

# Seleccionar filas donde Precio sea menor a 0.5.
# Nota: en tu dataset en inglés el precio mínimo ronda los 30, por lo que este filtro puede salir vacío.
precios_menores_05 = datos[datos["Precio"] < 0.5]

print("\nPRECIOS MENORES A 0.5")
print("Cantidad de registros:", len(precios_menores_05))
display(precios_menores_05)

# query() permite escribir las condiciones dentro de un texto.
# Adaptamos la consulta para buscar una categoría existente en tu dataset como 'Beauty' o 'Clothing'
categoria_busqueda = "Beauty" if "Beauty" in datos["Producto"].values else "Manzanas"
ventas_filtradas = datos.query(
    f"Producto == '{categoria_busqueda}' and Ventas > 3",
    engine="python"
)

print(f"\n{categoria_busqueda.upper()} CON VENTAS MAYORES A 3")
print("Cantidad de registros:", len(ventas_filtradas))
display(ventas_filtradas)


# --------------------------------------------------
# 6. SELECCIÓN DE FILAS Y COLUMNAS
# --------------------------------------------------

# Seleccionar únicamente las columnas Producto y Ventas.
print("\nCOLUMNAS PRODUCTO Y VENTAS")
display(datos[["Producto", "Ventas"]])

# loc selecciona por etiqueta e incluye ambos límites.
print("\nÍNDICES DEL 5 AL 10: PRODUCTO Y TIENDA")
display(datos.loc[5:10, ["Producto", "Tienda"]])

# iloc selecciona por posición y no incluye el límite final.
print("\nPRIMERAS 5 FILAS Y PRIMERAS 3 COLUMNAS")
display(datos.iloc[:5, :3])

print("\nAnálisis exploratorio finalizado.")

Selecciona el archivo retail_sales_dataset.csv o retail_sales.csv.


Saving retail_sales_dataset.csv to retail_sales_dataset.csv

Se detectó el dataset en inglés. Adaptando columnas...

PRIMERAS 10 FILAS


,Transaction ID,Date,Customer ID,Gender,Age,Producto,Ventas,Precio,Total Amount,Tienda
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,Tienda Online
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,Tienda Online
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,Tienda Online
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,Tienda Online
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,Tienda Online
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30,Tienda Online
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50,Tienda Online
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100,Tienda Online
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600,Tienda Online
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200,Tienda Online



ÚLTIMAS 5 FILAS


,Transaction ID,Date,Customer ID,Gender,Age,Producto,Ventas,Precio,Total Amount,Tienda
995,996,2023-05-16,CUST996,Male,62,Clothing,1,50,50,Tienda Online
996,997,2023-11-17,CUST997,Male,52,Beauty,3,30,90,Tienda Online
997,998,2023-10-29,CUST998,Female,23,Beauty,4,25,100,Tienda Online
998,999,2023-12-05,CUST999,Female,36,Electronics,3,50,150,Tienda Online
999,1000,2023-04-12,CUST1000,Male,47,Electronics,4,30,120,Tienda Online



INFORMACIÓN GENERAL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Transaction ID  1000 non-null   int64 
 1   Date            1000 non-null   object
 2   Customer ID     1000 non-null   object
 3   Gender          1000 non-null   object
 4   Age             1000 non-null   int64 
 5   Producto        1000 non-null   object
 6   Ventas          1000 non-null   int64 
 7   Precio          1000 non-null   int64 
 8   Total Amount    1000 non-null   int64 
 9   Tienda          1000 non-null   object
dtypes: int64(5), object(5)
memory usage: 78.3+ KB

TIPOS DE DATOS ORIGINALES
Transaction ID     int64
Date              object
Customer ID       object
Gender            object
Age                int64
Producto          object
Ventas             int64
Precio             int64
Total Amount       int64
Tienda            object
dtype: object

VA

... (10 outputs left out)

,Producto,Tienda
5,Beauty,Tienda Online
6,Clothing,Tienda Online
7,Electronics,Tienda Online
8,Electronics,Tienda Online
9,Clothing,Tienda Online
10,Clothing,Tienda Online



PRIMERAS 5 FILAS Y PRIMERAS 3 COLUMNAS


,Transaction ID,Date,Customer ID
0,1,2023-11-24,CUST001
1,2,2023-02-27,CUST002
2,3,2023-01-13,CUST003
3,4,2023-05-21,CUST004
4,5,2023-05-06,CUST005



Análisis exploratorio finalizado.
